# Level-1: subset flux columns

Reads the per-version Parquet files produced in
[notebook 01](01_read_fluxes_to_parquet.ipynb), keeps only a defined list of
columns, and writes the reduced tables back out as Parquet. This keeps the
downstream comparison focused on the variables of interest (fluxes and
time-lag diagnostics) instead of the full ~483-column EddyPro output.

- **Input:** `data/01-eddypro_fluxes_level-1_parquet/` — full per-version tables.
- **Output:** `data/02-eddypro_fluxes_level-1_parquet_subsets/` — column subsets.

The set of columns kept is defined explicitly in `KEEP_COLS` below so it stays
auditable alongside the [processing versions](../docs/processing-versions.md).

## Imports

In [26]:
from datetime import datetime
from pathlib import Path

from diive.core.io.files import load_parquet, save_parquet

NB_START = datetime.now()  # notebook start time (reported in the last cell)

## Select columns

In [27]:
# Columns to keep in each subset. Edit this list as the analysis requires.
KEEP_COLS = [
    # Fluxes
    "FN2O",
    "FCH4",
    # N2O time-lag diagnostics
    "N2O_TLAG_USED",
    # CH4 time-lag diagnostics
    "CH4_TLAG_USED",
]
print(f"{len(KEEP_COLS)} columns requested.")

4 columns requested.


## Configuration

In [28]:
# Input / output folders (relative to the notebooks/ directory).
INDIR = Path("../data/01-eddypro_fluxes_level-1_parquet")
OUTDIR = Path("../data/02-eddypro_fluxes_level-1_parquet_subsets")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Discover the input Parquet files.
parquet_files = sorted(INDIR.glob("*.parquet"))
print(f"Found {len(parquet_files)} Parquet file(s):")
for f in parquet_files:
    print(f"  {f.name}")

Found 6 Parquet file(s):
  LGR-1.parquet
  LGR-2R.parquet
  LGR-3.parquet
  QCL-1.parquet
  QCL-2R.parquet
  QCL-3.parquet


## Subset each file and save

For each version: load the full table, keep `KEEP_COLS` (preserving the
timestamp index), warn about any requested columns that are absent, and save the
subset under the same version-code filename.

In [29]:
saved = {}
for f in parquet_files:
    code = f.stem  # version code, e.g. 'LGR-1'
    print(f"\n=== {code} ===")

    df = load_parquet(filepath=str(f))

    present = [c for c in KEEP_COLS if c in df.columns]
    missing = [c for c in KEEP_COLS if c not in df.columns]
    if missing:
        print(f"  WARNING: {len(missing)} requested column(s) not found: {missing}")

    subset = df[present]
    print(f"  kept {subset.shape[1]} / {len(KEEP_COLS)} columns, {subset.shape[0]} rows")

    filepath = save_parquet(filename=code, data=subset, outpath=str(OUTDIR))
    saved[code] = filepath


=== LGR-1 ===


> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet (0.075 seconds).

  kept 4 / 4 columns, 7805 rows


> Saved file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-1.parquet (0.015 seconds).


=== LGR-2R ===


...

> Saved file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-2R.parquet (0.019 seconds).


=== QCL-3 ===


> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-3.parquet (0.125 seconds).

  kept 4 / 4 columns, 20790 rows


> Saved file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-3.parquet (0.041 seconds).

## Verify

Reload one subset to confirm the round-trip and the retained columns.

In [30]:
print("Saved subset files:")
for code, path in saved.items():
    print(f"  {code}: {path}")

check = load_parquet(filepath=next(iter(saved.values())))
print(f"\nColumns ({check.shape[1]}): {list(check.columns)}")
check.describe()

Saved subset files:
  LGR-1: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-1.parquet
  LGR-2R: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-2R.parquet
  LGR-3: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-3.parquet
  QCL-1: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-1.parquet
  QCL-2R: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-2R.parquet
  QCL-3: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-3.parquet


> Loaded .parquet file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-1.parquet (0.010 seconds).


Columns (4): ['FN2O', 'FCH4', 'N2O_TLAG_USED', 'CH4_TLAG_USED']


,FN2O,FCH4,N2O_TLAG_USED,CH4_TLAG_USED
count,7731.000000,7731.000000,7731.000000,7731.000000
mean,1.821938,15.252026,3.369681,4.024648
std,9.197670,221.503720,3.196333,3.900663
min,-374.735000,-7194.450000,-1.000000,-1.000000
25%,0.103841,-11.579100,1.600000,1.300000
50%,0.723384,4.498810,1.850000,2.250000
75%,2.550460,27.070200,5.500000,8.200000
max,205.037000,5268.080000,10.000000,10.000000


## Runtime

In [31]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")

Start:    2026-06-04 14:17:31
End:      2026-06-04 14:17:34
Runtime:  0:00:03.328534
